# I94 Immigration Data Enrichment and ETL Pipeline
### Data Engineering Capstone Project

#### Project Summary
- The goal of the project is to build a database to get a in-depth view of immigration in the US which would help us understand the various underlying patterns immigrants follow and their rationale behind a vertical shift from various parts of the world.
- The project will serve as a data source for data analysts to find if temperature drives immigration to certain regions of the US and data scientists can build a model to forecast the number of immigrants according to their business requirement.

The project follows the follow steps:
* Step 1: Scope the Project and Gather Data
* Step 2: Explore and Assess the Data
* Step 3: Define the Data Model
* Step 4: Run ETL to Model the Data
* Step 5: Complete Project Write Up

In [1]:
# import libraries 
import pandas as pd
import psycopg2
import os
from pyspark.sql.functions import udf
from pyspark.sql import SparkSession


# env var set-up
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["PATH"] = "/opt/conda/bin:/opt/spark-2.4.3-bin-hadoop2.7/bin:/opt/conda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/lib/jvm/java-8-openjdk-amd64/bin"
os.environ["SPARK_HOME"] = "/opt/spark-2.4.3-bin-hadoop2.7"
os.environ["HADOOP_HOME"] = "/opt/spark-2.4.3-bin-hadoop2.7"
spark = SparkSession.builder.getOrCreate()
print("Completed, Environment set-up successful")

Completed, Environment set-up successful


### Step 1: Scope the Project and Gather Data

#### Scope 

- The objective is to create a multi-purpose database which can serve as an inference source or can be used to find deeper insights about US immigrations. We're going to achieve this by enriching the I94 Immigration Data(US) with supporting datasets like World Temperature Data and Airport Code Table:

#### Describe and Gather Data 
1. **I94 Immigration Data**: This data comes from the US National Tourism and Trade Office. [This](https://www.trade.gov/national-travel-and-tourism-office) is where the data comes from. 

2. **World Temperature Data**: This dataset came from Kaggle. You can read more about it [here](https://www.kaggle.com/berkeleyearth/climate-change-earth-surface-temperature-data).

3. **Airport Code Table**: This is a simple table of airport codes and corresponding cities. It comes from [here](https://datahub.io/core/airport-codes#data).

In [2]:
# reading I94 immigration data

immigration_data =spark.read.load('./sas_data')
immigration_data.show(5)

+---------+------+------+------+------+-------+-------+-------+-------+-------+------+-------+-----+--------+--------+-----+-------+-------+-------+-------+-------+--------+------+------+-------+--------------+-----+--------+
|    cicid| i94yr|i94mon|i94cit|i94res|i94port|arrdate|i94mode|i94addr|depdate|i94bir|i94visa|count|dtadfile|visapost|occup|entdepa|entdepd|entdepu|matflag|biryear| dtaddto|gender|insnum|airline|        admnum|fltno|visatype|
+---------+------+------+------+------+-------+-------+-------+-------+-------+------+-------+-----+--------+--------+-----+-------+-------+-------+-------+-------+--------+------+------+-------+--------------+-----+--------+
|5748517.0|2016.0|   4.0| 245.0| 438.0|    LOS|20574.0|    1.0|     CA|20582.0|  40.0|    1.0|  1.0|20160430|     SYD| null|      G|      O|   null|      M| 1976.0|10292016|     F|  null|     QF|9.495387003E10|00011|      B1|
|5748518.0|2016.0|   4.0| 245.0| 438.0|    LOS|20574.0|    1.0|     NV|20591.0|  32.0|    1.0|  

In [3]:
# reading world temperature data

temperature_data = spark.read.format("csv").option("header", "true").load("../../data2/GlobalLandTemperaturesByCity.csv")
temperature_data.show(5)

+----------+------------------+-----------------------------+-----+-------+--------+---------+
|        dt|AverageTemperature|AverageTemperatureUncertainty| City|Country|Latitude|Longitude|
+----------+------------------+-----------------------------+-----+-------+--------+---------+
|1743-11-01|             6.068|           1.7369999999999999|Århus|Denmark|  57.05N|   10.33E|
|1743-12-01|              null|                         null|Århus|Denmark|  57.05N|   10.33E|
|1744-01-01|              null|                         null|Århus|Denmark|  57.05N|   10.33E|
|1744-02-01|              null|                         null|Århus|Denmark|  57.05N|   10.33E|
|1744-03-01|              null|                         null|Århus|Denmark|  57.05N|   10.33E|
+----------+------------------+-----------------------------+-----+-------+--------+---------+
only showing top 5 rows



In [4]:
# reading airport code data

airport_data = pd.read_csv('airport-codes_csv.csv')
airport_data.head()

,ident,type,name,elevation_ft,continent,iso_country,iso_region,municipality,gps_code,iata_code,local_code,coordinates
0,00A,heliport,Total Rf Heliport,11.0,NaN,US,US-PA,Bensalem,00A,NaN,00A,"-74.93360137939453, 40.07080078125"
1,00AA,small_airport,Aero B Ranch Airport,3435.0,NaN,US,US-KS,Leoti,00AA,NaN,00AA,"-101.473911, 38.704022"
2,00AK,small_airport,Lowell Field,450.0,NaN,US,US-AK,Anchor Point,00AK,NaN,00AK,"-151.695999146, 59.94919968"
3,00AL,small_airport,Epps Airpark,820.0,NaN,US,US-AL,Harvest,00AL,NaN,00AL,"-86.77030181884766, 34.86479949951172"
4,00AR,closed,Newport Hospital & Clinic Heliport,237.0,NaN,US,US-AR,Newport,NaN,NaN,NaN,"-91.254898, 35.6087"


### Step 2: Explore and Assess the Data

#### Data Exploration, Cleaning and Manipulation

1. I94 immigration data - After observing the "I94_SAS_Labels_Descriptions" file, there are around 80 irregular port ids in the I94 data.First, we clean the data with irregular port ids then also drop columns and rows with null values to improve the overall data quality.

2. Global Temperature Data - Since the data is global, we filter the data according to our need i.e we filter for "United States" and filter out the null values from the data. 

3. Airport Code Table - We filter out the null values and merge the data with ports data to add some extra information. We take a subset of important columns and store it in a dataframe.

### I94 immigration data

In [5]:
# Performing cleaning tasks on I94 immigration data

import pandas as pd

with open("./I94_SAS_Labels_Descriptions.SAS") as f:
    file_data = f.readlines()

# removes whitespaces in the file
file_data = [x.strip() for x in file_data]

# ports data starts from index 302
ports_data = file_data[302:962]

# cleaning ports data
ports_data_cleaned = [port.split("=") for port in ports_data]

# port codes
port_codes = [x[0].replace("'","").strip() for x in ports_data_cleaned]

# port locations
port_locations = [x[1].replace("'","").strip() for x in ports_data_cleaned]

# port cities
port_cities = [x.split(",")[0] for x in port_locations]

# port states
port_states = [x.split(",")[-1] for x in port_locations]

port_data_dict = {"port_code" : port_codes, 
                  "port_city": port_cities, 
                  "port_state": port_states,
                  "port_loc":port_locations}

port_df = pd.DataFrame(port_data_dict)
port_df.head()

,port_code,port_city,port_state,port_loc
0,ALC,ALCAN,AK,"ALCAN, AK"
1,ANC,ANCHORAGE,AK,"ANCHORAGE, AK"
2,BAR,BAKER AAF - BAKER ISLAND,AK,"BAKER AAF - BAKER ISLAND, AK"
3,DAC,DALTONS CACHE,AK,"DALTONS CACHE, AK"
4,PIZ,DEW STATION PT LAY DEW,AK,"DEW STATION PT LAY DEW, AK"


In [6]:
# cleaning the data using the data desc
print("First port:",port_df['port_city'].values[0])
print("Last port:",port_df['port_city'].values[-1])

# pattern observed in the Label Description file
ports_df_irr = port_df[port_df["port_city"] == port_df["port_state"]]
print("Shape:",ports_df_irr.shape)

First port: ALCAN
Last port: No PORT Code (OSN)
Shape: (77, 4)


In [7]:
print(port_df.shape)
port_df_valid = port_df[~(port_df['port_code'].isin(set(ports_df_irr["port_code"].tolist())))]
print(port_df_valid.shape)

(660, 4)
(583, 4)


In [8]:
# add desc to fn
def spark_shape(df):
    ''' 
    Function to return the shape of a spark df"
    args:
        df(dataframe):
    returns:
        no_rows: no. of rows in the df
        no_cols: no. of columns in the df
    '''
    no_rows = df.count()
    no_cols = len(df.columns)
    
    return (no_rows,no_cols)

In [9]:
# drop all irregular ports from i94 data
print("No. of Rows before cleaning:",spark_shape(immigration_data))
immigration_data_filtered = immigration_data.filter(~(immigration_data.i94port.isin(set(ports_df_irr["port_code"].tolist()))))
print("No. of Rows after cleaning:",spark_shape(immigration_data_filtered))

No. of Rows before cleaning: (3096313, 28)
No. of Rows after cleaning: (2995590, 28)


In [10]:
# cols where majority of the data is null
immigration_data_filtered_new = immigration_data_filtered.drop("insnum", "entdepu", "occup", "visapost")
immigration_data_filtered_new.printSchema()

root
 |-- cicid: double (nullable = true)
 |-- i94yr: double (nullable = true)
 |-- i94mon: double (nullable = true)
 |-- i94cit: double (nullable = true)
 |-- i94res: double (nullable = true)
 |-- i94port: string (nullable = true)
 |-- arrdate: double (nullable = true)
 |-- i94mode: double (nullable = true)
 |-- i94addr: string (nullable = true)
 |-- depdate: double (nullable = true)
 |-- i94bir: double (nullable = true)
 |-- i94visa: double (nullable = true)
 |-- count: double (nullable = true)
 |-- dtadfile: string (nullable = true)
 |-- entdepa: string (nullable = true)
 |-- entdepd: string (nullable = true)
 |-- matflag: string (nullable = true)
 |-- biryear: double (nullable = true)
 |-- dtaddto: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- airline: string (nullable = true)
 |-- admnum: double (nullable = true)
 |-- fltno: string (nullable = true)
 |-- visatype: string (nullable = true)



In [11]:
# dropping null values from the df

print("Before nulls are dropped:",spark_shape(immigration_data_filtered_new))
immigration_data_filtered_nd = immigration_data_filtered_new.na.drop()
print("After nulls are dropped:",spark_shape(immigration_data_filtered_nd))

Before nulls are dropped: (2995590, 24)
After nulls are dropped: (2306750, 24)


### Global Temperature Data

In [12]:
us_temperature_data = temperature_data.filter(temperature_data.Country == "United States")
us_temperature_data.show(5)

+----------+------------------+-----------------------------+-------+-------------+--------+---------+
|        dt|AverageTemperature|AverageTemperatureUncertainty|   City|      Country|Latitude|Longitude|
+----------+------------------+-----------------------------+-------+-------------+--------+---------+
|1820-01-01|2.1010000000000004|                        3.217|Abilene|United States|  32.95N|  100.53W|
|1820-02-01|             6.926|                        2.853|Abilene|United States|  32.95N|  100.53W|
|1820-03-01|            10.767|                        2.395|Abilene|United States|  32.95N|  100.53W|
|1820-04-01|17.988999999999994|                        2.202|Abilene|United States|  32.95N|  100.53W|
|1820-05-01|            21.809|                        2.036|Abilene|United States|  32.95N|  100.53W|
+----------+------------------+-----------------------------+-------+-------------+--------+---------+
only showing top 5 rows



In [13]:
us_temperature_data_nd = us_temperature_data.na.drop()
us_temperature_data_nd.show(2)

+----------+------------------+-----------------------------+-------+-------------+--------+---------+
|        dt|AverageTemperature|AverageTemperatureUncertainty|   City|      Country|Latitude|Longitude|
+----------+------------------+-----------------------------+-------+-------------+--------+---------+
|1820-01-01|2.1010000000000004|                        3.217|Abilene|United States|  32.95N|  100.53W|
|1820-02-01|             6.926|                        2.853|Abilene|United States|  32.95N|  100.53W|
+----------+------------------+-----------------------------+-------+-------------+--------+---------+
only showing top 2 rows



In [14]:
print("Before nulls are dropped:",spark_shape(us_temperature_data))
print("After nulls are dropped:",spark_shape(us_temperature_data_nd))

Before nulls are dropped: (687289, 7)
After nulls are dropped: (661524, 7)


### Airport Code Table

In [15]:
# dropping nulls and merging the data

airport_data_nd = airport_data.dropna(subset=['iata_code'])
print(airport_data_nd.shape)

airport_data_merged = airport_data_nd.merge(port_df_valid, left_on="iata_code", right_on="port_code",how='inner')
print(airport_data_merged.shape)
airport_data_merged.head()

(9189, 12)
(510, 16)


,ident,type,name,elevation_ft,continent,iso_country,iso_region,municipality,gps_code,iata_code,local_code,coordinates,port_code,port_city,port_state,port_loc
0,57A,seaplane_base,Tokeen Seaplane Base,NaN,NaN,US,US-AK,Tokeen,57A,TKI,57A,"-133.32699585, 55.9370994568",TKI,TOKEEN,AK,"TOKEEN, AK"
1,89NY,small_airport,Maxson Airfield,340.0,NaN,US,US-NY,Alexandria Bay,89NY,AXB,89NY,"-75.90034, 44.312002",AXB,ALEXANDRIA BAY,NY,"ALEXANDRIA BAY, NY"
2,AGGF,small_airport,Fera/Maringe Airport,NaN,OC,SB,SB-IS,Fera Island,AGGF,FRE,NaN,"159.576996, -8.1075",FRE,FRESNO,CA,"FRESNO, CA"
3,ANZ,small_airport,Angus Downs Airport,1724.0,OC,AU,AU-NT,Angus Downs Station,NaN,ANZ,NaN,"132.2748, -25.0325",ANZ,ANZALDUAS,TX,"ANZALDUAS, TX"
4,AU-BCK,closed,[Duplicate] Bolwarra Airport,NaN,OC,AU,AU-QLD,Bolwarra,NaN,BCK,NaN,"144.169006348, -17.388299942",BCK,BUCKPORT,ME,"BUCKPORT, ME"


In [16]:
# taking a subset of important columns

airport_data_merged_filtered = airport_data_merged[['iata_code','name','type','local_code','coordinates','port_city','port_state','elevation_ft']]
airport_data_merged_filtered.head()

,iata_code,name,type,local_code,coordinates,port_city,port_state,elevation_ft
0,TKI,Tokeen Seaplane Base,seaplane_base,57A,"-133.32699585, 55.9370994568",TOKEEN,AK,NaN
1,AXB,Maxson Airfield,small_airport,89NY,"-75.90034, 44.312002",ALEXANDRIA BAY,NY,340.0
2,FRE,Fera/Maringe Airport,small_airport,NaN,"159.576996, -8.1075",FRESNO,CA,NaN
3,ANZ,Angus Downs Airport,small_airport,NaN,"132.2748, -25.0325",ANZALDUAS,TX,1724.0
4,BCK,[Duplicate] Bolwarra Airport,closed,NaN,"144.169006348, -17.388299942",BUCKPORT,ME,NaN


In [17]:
airport_data_merged_filtered[['local_code','elevation_ft']].isnull().sum()

local_code      227
elevation_ft     21
dtype: int64

In [18]:
airport_data_merged_filtered['local_code'].fillna('-',inplace=True)
airport_data_merged_filtered['elevation_ft'].fillna(0.0,inplace=True)
airport_data_merged_filtered[['local_code','elevation_ft']].isnull().sum()

/opt/conda/lib/python3.6/site-packages/pandas/core/generic.py:5434: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  self._update_inplace(new_data)


local_code      0
elevation_ft    0
dtype: int64

### Step 3: Define the Data Model

#### 3.1 Conceptual Data Model

- Tables 
| Table | Type | Description | Columns |
| --- | --- | --- | --- |
| immigration | Fact Table | stores i94 immigrations data | cicid - i94yr - i94mon - i94cit - i94res - i94port - arrdate - i94mode - i94addr -depdate - i94bir - i94visa - count - dtadfile - entdepa - entdepd - matflag - biryear - dtaddto - gender - airline - admnum - fltno - visatype | 
| temperature | Dimension Table | stores temperature data | dt - AverageTemperature - AverageTemperatureUncertainty - City - Country -            Latitude - Longitude - i94port |
| airport | Dimension Table | stores i94 airport data | iata_code - name - type - local_code - coordinates - port_city - port_state -            elevation_ft |

#### 3.2 Mapping Out Data Pipelines

1. Global Temperature Data - We add port codes to the temperature data using a spark udf and then drop all the nulls values.
2. Airport Code Table - We convert the filtered dataframe to a spark dataframe.
3. Finally we complete the pipeline by creating parquet files on appropriate partition keys.

### Step 4: Run Pipelines to Model the Data 
#### 4.1 Create the data model
Build the data pipelines to create the data model.

### Global Temperature Data

In [19]:
# creating function to add port code to temperature data

valid_port_list = list(port_df_valid['port_loc'].unique())

@udf()
def get_port(city):
    '''
    Function to return port code if given condition is true
    args:
        city(string): name of the city
    returns:
        port_code(string): code of the respective port
    '''
    
    for port_code in valid_port_list:
        if city.lower() in port_code.lower():
            return port_code

In [20]:
# adding the ports to temperature data

us_temperature_data_complete = us_temperature_data_nd.withColumn("i94port", get_port(us_temperature_data_nd.City))
us_temperature_data_complete.show(5)

+----------+------------------+-----------------------------+-------+-------------+--------+---------+-------+
|        dt|AverageTemperature|AverageTemperatureUncertainty|   City|      Country|Latitude|Longitude|i94port|
+----------+------------------+-----------------------------+-------+-------------+--------+---------+-------+
|1820-01-01|2.1010000000000004|                        3.217|Abilene|United States|  32.95N|  100.53W|   null|
|1820-02-01|             6.926|                        2.853|Abilene|United States|  32.95N|  100.53W|   null|
|1820-03-01|            10.767|                        2.395|Abilene|United States|  32.95N|  100.53W|   null|
|1820-04-01|17.988999999999994|                        2.202|Abilene|United States|  32.95N|  100.53W|   null|
|1820-05-01|            21.809|                        2.036|Abilene|United States|  32.95N|  100.53W|   null|
+----------+------------------+-----------------------------+-------+-------------+--------+---------+-------+
o

In [21]:
# dropping null values
us_temperature_data_complete_nd = us_temperature_data_complete.na.drop()
us_temperature_data_complete_nd.show(5)

+----------+------------------+-----------------------------+-----+-------------+--------+---------+---------+
|        dt|AverageTemperature|AverageTemperatureUncertainty| City|      Country|Latitude|Longitude|  i94port|
+----------+------------------+-----------------------------+-----+-------------+--------+---------+---------+
|1743-11-01|             3.209|           1.9609999999999999|Akron|United States|  40.99N|   80.95W|AKRON, OH|
|1744-04-01|            10.352|                        2.222|Akron|United States|  40.99N|   80.95W|AKRON, OH|
|1744-05-01|            15.487|                        1.867|Akron|United States|  40.99N|   80.95W|AKRON, OH|
|1744-06-01|              20.9|                        1.726|Akron|United States|  40.99N|   80.95W|AKRON, OH|
|1744-07-01|            22.111|           1.5290000000000001|Akron|United States|  40.99N|   80.95W|AKRON, OH|
+----------+------------------+-----------------------------+-----+-------------+--------+---------+---------+
o

In [22]:
print("Before nulls are dropped:",spark_shape(us_temperature_data_complete))
print("After nulls are dropped:",spark_shape(us_temperature_data_complete_nd))

Before nulls are dropped: (661524, 8)
After nulls are dropped: (313024, 8)


### Airport Code Table

In [23]:
# converts pandas df to spark df

airport_data_merged_spark_df = spark.createDataFrame(airport_data_merged_filtered)
airport_data_merged_spark_df.show(5)

+---------+--------------------+-------------+----------+--------------------+--------------+----------+------------+
|iata_code|                name|         type|local_code|         coordinates|     port_city|port_state|elevation_ft|
+---------+--------------------+-------------+----------+--------------------+--------------+----------+------------+
|      TKI|Tokeen Seaplane Base|seaplane_base|       57A|-133.32699585, 55...|        TOKEEN|        AK|         0.0|
|      AXB|     Maxson Airfield|small_airport|      89NY|-75.90034, 44.312002|ALEXANDRIA BAY|        NY|       340.0|
|      FRE|Fera/Maringe Airport|small_airport|         -| 159.576996, -8.1075|        FRESNO|        CA|         0.0|
|      ANZ| Angus Downs Airport|small_airport|         -|  132.2748, -25.0325|     ANZALDUAS|        TX|      1724.0|
|      BCK|[Duplicate] Bolwa...|       closed|         -|144.169006348, -1...|      BUCKPORT|        ME|         0.0|
+---------+--------------------+-------------+----------

In [25]:
# write I94 data to parquet files partitioned by i94port
immigration_data_filtered_nd.write.mode("append").partitionBy("i94port").parquet("./final_output/immigration.parquet")

In [26]:
# write temperature data to parquet files partitioned by i94port
us_temperature_data_complete_nd.write.mode("append").partitionBy("i94port").parquet("./final_output/temperature.parquet")

In [27]:
# write temperature data to parquet files partitioned by iata code
airport_data_merged_spark_df.write.mode("append").partitionBy("iata_code").parquet("./final_output/airport.parquet")

#### 4.2 Data Quality Checks
Explain the data quality checks you'll perform to ensure the pipeline ran as expected. These could include:
 * Integrity constraints on the relational database (e.g., unique key, data type, etc.)
 * Unit tests for the scripts to ensure they are doing the right thing
 * Source/Count checks to ensure completeness
1. Quality Check 1: Checks the number of rows and columns of the respective dataframes.
2. Quality Check 2: Checks the shape of the parquet files and the original data to ensure that parquet files are written successfully.

#### Quality Check 1

In [28]:
def quality_check_1(df,desc):
    '''
    checks the value counts of the spark dfs and returns respective logs
    args:
        df(dataframe): respective dataframe for data validation
        desc(string): appropriate description for the data
    returns:
        string result with no. of rows and columns
    '''
    
    rows = df.count()
    columns = len(df.columns)
    if (rows == 0 & columns==0):
        print("Data quality check failed for {} with zero records".format(desc))
    else:
        print("Data quality check passed for {} with {} rows and {} columns".format(desc, rows,columns))

In [29]:
quality_check_1(immigration_data_filtered_nd,"immigration table")

Data quality check passed for immigration table with 2306750 rows and 24 columns


In [30]:
quality_check_1(us_temperature_data_complete_nd,"temperature table")

Data quality check passed for temperature table with 313024 rows and 8 columns


In [31]:
quality_check_1(airport_data_merged_spark_df,"airport table")

Data quality check passed for airport table with 510 rows and 8 columns


#### Quality Check 2

In [32]:
def quality_check_2(df,data_path,desc):
    '''
    Function to check the shape of parquet files with the source files
    args:
        df(dataframe): respective dataframe for data validation
        data_path(string): path to parquet file
        desc(string): appropriate description for the data
    returns:
        string result for the respective condition
    '''
    
    data = spark.read.parquet(data_path)
    data_shape = spark_shape(data)
    
    df_shape = spark_shape(df)
    
    if df_shape == data_shape:
        print("Data quality check passed for {}".format(desc))
    else:
        print("Data quality check failed for {}".format(desc))

In [33]:
quality_check_2(immigration_data_filtered_nd,"./final_output/immigration.parquet","immigration table")

Data quality check passed for immigration table


In [34]:
quality_check_2(us_temperature_data_complete_nd,"./final_output/temperature.parquet","temperature table")

Data quality check passed for temperature table


In [35]:
quality_check_2(airport_data_merged_spark_df,"./final_output/airport.parquet","airport table")

Data quality check passed for airport table


#### 4.3 Data dictionary 
Create a data dictionary for your data model. For each field, provide a brief description of what the data is and where it came from. You can include the data dictionary in the notebook or in a separate file.

**1. Immigration Data :  I94 immigration data**
- cicid
- i94yr
- i94mon
- i94cit
- i94res
- i94port
- arrdate
- i94mode
- i94addr
- depdate
- i94bir
- i94visa
- count
- dtadfile
- entdepa
- entdepd
- matflag
- biryear
- dtaddto
- gender
- airline
- admnum
- fltno
- visatype

**2. Temperature Data : Temprature Data data with port codes.**
- dt 
- AverageTemperature 
- AverageTemperatureUncertainty 
- City 
- Country 
- Latitude 
- Longitude 
- i94port

**3. Airport Data : Aiport data with port,city and states info.**
- iata_code 
- name 
- type 
- local_code 
- coordinates 
- port_city 
- port_state 
- elevation_ft

#### Step 5: Complete Project Write Up

##### Clearly state the rationale for the choice of tools and technologies for the project.
- The combination of Spark and Pandas gave me the flexibility over data manipulations and exploration without worrying about big data processing. Both tools have similar functionalities and are easy to use which made data transformations less complex.
- Currently, to save loading time the data is not stored to a cloud storage but when the need arises we can set-up a s3 bucket or redshift to store the parquet files.

#### Propose how often the data should be updated and why.
- A monthly data refresh makes sense here since the volume of monthly data will be easy to handle and would be a logical choice with respect to our different data sources.

#### Write a description of how you would approach the problem differently under the following scenarios:
 1. The data was increased by 100x.
     - Improving our current spark processes using Amazon EMR and distributing the workload to multiple nodes would enable us to handle the exponential growth in data.
     
 2. The data populates a dashboard that must be updated on a daily basis by 7am every day.
     - We can set-up airflow to schedule and orchestrate our data pipelines everyday which can also keep us updated about our pipeline execution over an email.
     
 3. The database needed to be accessed by 100+ people.
     - Setting up a Redshift Cluster for our data can eliminate this issue.

#### Sample Query: Finding the Gender wise Visa type distribution

In [36]:
# sample query
immigration_data_filtered_nd.groupby('gender').pivot('visatype').count().show()

+------+------+------+----+----+----+-----+-----+----+----+-----+----+----+----+----+----+------+------+
|gender|    B1|    B2|  CP| CPL|  E1|   E2|   F1|  F2| GMB|  GMT|   I|  I1|  M1|  M2| SBP|    WB|    WT|
+------+------+------+----+----+----+-----+-----+----+----+-----+----+----+----+----+----+------+------+
|     F| 41095|473290| 404|null| 669| 3757|11183| 693|  40|35817| 667|  51| 224|  23|   1| 34476|515700|
|     M|125970|394544| 462|   8|2181|10733|12025| 808|  87|31074|1921| 134| 419|   4|   1|137936|470096|
|     U|     1|     5|null|null|null| null| null|null|null| null|null|null|null|null|null|  null|     1|
|     X|  null|  null|  17|null|null| null| null|null|null|  229|null|null|null|null|null|  null|     4|
+------+------+------+----+----+----+-----+-----+----+----+-----+----+----+----+----+----+------+------+



In [ ]:
# end